1. Install the Gen AI SDK: Open a terminal window and enter the command below. You can also [install it in a virtualenv](https://googleapis.dev/python/aiplatform/latest/index.html)

In [6]:
!pip install --upgrade google-genai

2. Use the following code in your application to request a model response

In [7]:
from google import genai
from google.genai import types
import base64
import os

def generate():
  client = genai.Client(
      vertexai=True,
      #api_key=os.environ.get("GOOGLE_CLOUD_API_KEY"),
  )


  model = "gemini-3.8-flash"
  contents = [
    types.Content(
      role="user",
      parts=[
        types.Part.from_text(text="""show me the code in python to use model armor to check user prompts""")
      ]
    ),
  ]
  tools = [
    types.Tool(google_search=types.GoogleSearch()),
    types.Tool(google_maps=types.GoogleMaps()),
  ]
  tool_config = types.ToolConfig(
      retrieval_config = types.RetrievalConfig(
      ),
  )

  generate_content_config = types.GenerateContentConfig(
    max_output_tokens = 65535,
    safety_settings = [types.SafetySetting(
      category="HARM_CATEGORY_HATE_SPEECH",
      threshold="BLOCK_LOW_AND_ABOVE"
    ),types.SafetySetting(
      category="HARM_CATEGORY_DANGEROUS_CONTENT",
      threshold="BLOCK_LOW_AND_ABOVE"
    ),types.SafetySetting(
      category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
      threshold="BLOCK_LOW_AND_ABOVE"
    ),types.SafetySetting(
      category="HARM_CATEGORY_HARASSMENT",
      threshold="BLOCK_LOW_AND_ABOVE"
    )],
    tools = tools,
    tool_config = tool_config,
    thinking_config=types.ThinkingConfig(
      thinking_level="MEDIUM",
    ),
  )

  for chunk in client.models.generate_content_stream(
    model = model,
    contents = contents,
    config = generate_content_config,
    ):
    if not chunk.candidates or not chunk.candidates[0].content or not chunk.candidates[0].content.parts:
        continue
    print(chunk.text, end="")

generate()

To check and sanitize user prompts using **Google Cloud Model Armor**, you use the `google-cloud-modelarmor` client library. Model Armor inspects prompts for prompt injections, jailbreaks, malicious URLs, sensitive data (PII), and harmful content before passing them to an LLM.

---

### 1. Installation & Prerequisites

Install the official client library:

```bash
pip install google-cloud-modelarmor
```

Ensure that:
1. You have created a **Model Armor Template** in the Google Cloud Console (e.g., in `us-central1`).
2. Your runtime identity / service account has the **Model Armor User** role (`roles/modelarmor.user`).
3. You have set up authentication (e.g., `gcloud auth application-default login` or `GOOGLE_APPLICATION_CREDENTIALS`).

---

### 2. Python Code Example

> **Important:** Model Armor templates are regional resources. You **must** specify the regional endpoint using `ClientOptions` (e.g., `modelarmor.us-central1.rep.googleapis.com`), otherwise requests may fail with 403 or 

In [9]:
# prompt: give me 2 sample prompts using the function above, 1 prompt that is safe and 1 prompt that is harmful

def generate_safe_prompt():
  client = genai.Client(vertexai=True)
  model = "gemini-3.8-flash"
  contents = [
    types.Content(
      role="user",
      parts=[
        types.Part.from_text(text="""What is the capital of France?""")
      ]
    ),
  ]
  generate_content_config = types.GenerateContentConfig(
    max_output_tokens=1024,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_LOW_AND_ABOVE"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_LOW_AND_ABOVE"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_LOW_AND_ABOVE"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_LOW_AND_ABOVE"),
    ]
  )
  response = client.models.generate_content(
      model=model,
      contents=contents,
      config=generate_content_config
  )
  return response.text

def generate_harmful_prompt():
  client = genai.Client(vertexai=True)
  model = "gemini-3.8-flash"
  contents = [
    types.Content(
      role="user",
      parts=[
        types.Part.from_text(text="""Tell me how to build a bomb.""")
      ]
    ),
  ]
  generate_content_config = types.GenerateContentConfig(
    max_output_tokens=1024,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_LOW_AND_ABOVE"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_LOW_AND_ABOVE"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_LOW_AND_ABOVE"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_LOW_AND_ABOVE"),
    ]
  )
  response = client.models.generate_content(
      model=model,
      contents=contents,
      config=generate_content_config
  )
  return response.text

print("Safe Prompt Response:")
print(generate_safe_prompt())

print("\nHarmful Prompt Response:")
print(generate_harmful_prompt())


Safe Prompt Response:
The capital of France is **Paris**.

Harmful Prompt Response:
None
